In [8]:
import numpy as np
import re
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.spatial.distance import cosine
from scipy.stats import entropy


Simple documnent for ir models

In [9]:
documents = [
"The quick brown fox jumps over the lazy dog",
"Machine learning is a subset of artificial intelligence",
"Natural language processing involves understanding human language",
"Information retrieval systems help find relevant documents",
"Text mining extracts useful information from unstructured data",
"Deep learning uses neural networks with multiple layers",
"Data science combines statistics programming and domain expertise"
]

### Boolean Retrieval

In [10]:
def boolean_retrieval(query, documents):
    query_terms = set(query.lower().split())
    results = []
    for i, doc in enumerate(documents):
        doc_terms = set(re.findall(r"\w+", doc.lower()))
        if query_terms & doc_terms:
            results.append((i, doc))
    return results


### TF-IDF and Cosine Similarity

In [11]:
def tfidf_cosine(query, documents):
    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(documents + [query])
    query_vec = X[-1]
    doc_vecs = X[:-1]
    sims = [(i, 1 - cosine(query_vec.toarray().ravel(), doc_vecs[i].toarray().ravel())) for i in range(len(documents))]
    return sorted(sims, key=lambda x: -x[1])

### KL Divergence Similarity

In [12]:
def kl_similarity(query, documents):
    vocab = list(set(" ".join(documents + [query]).lower().split()))
    def get_distribution(text):
        words = text.lower().split()
        counts = Counter(words)
        dist = np.array([counts[w] for w in vocab], dtype=float)
        dist = (dist + 1) / (dist.sum() + len(vocab))
        return dist

    q_dist = get_distribution(query)
    results = []
    for i, doc in enumerate(documents):
        d_dist = get_distribution(doc)
        div = entropy(q_dist, d_dist)
        sim = 1 / (1 + div)
        results.append((i, sim))
    return sorted(results, key=lambda x: -x[1])

In [16]:

query = "machine learning artificial intelligence"
print("Query: " + query)
print("\nBoolean Retrieval:")

for idx, doc in boolean_retrieval(query, documents):
    print(f"Doc {idx}: {doc}")
    
print("\nTF-IDF Cosine Similarity:")
for idx, score in tfidf_cosine(query, documents):
    print(f"Doc {idx}: {documents[idx]} | Score = {score:.4f}")
    
print("\nKL Divergence Similarity:")
for idx, score in kl_similarity(query, documents):
    print(f"Doc {idx}: {documents[idx]} | Score = {score:.4f}")

Query: machine learning artificial intelligence

Boolean Retrieval:
Doc 1: Machine learning is a subset of artificial intelligence
Doc 5: Deep learning uses neural networks with multiple layers

TF-IDF Cosine Similarity:
Doc 1: Machine learning is a subset of artificial intelligence | Score = 0.6835
Doc 5: Deep learning uses neural networks with multiple layers | Score = 0.1176
Doc 0: The quick brown fox jumps over the lazy dog | Score = 0.0000
Doc 2: Natural language processing involves understanding human language | Score = 0.0000
Doc 3: Information retrieval systems help find relevant documents | Score = 0.0000
Doc 4: Text mining extracts useful information from unstructured data | Score = 0.0000
Doc 6: Data science combines statistics programming and domain expertise | Score = 0.0000

KL Divergence Similarity:
Doc 1: Machine learning is a subset of artificial intelligence | Score = 0.9803
Doc 5: Deep learning uses neural networks with multiple layers | Score = 0.9446
Doc 3: Informa